# GC-LSTM-GhostNet - Step 4 sampled training smoke test

Train GCN â†’ LSTM â†’ temporal attention â†’ GhostNet with leakage-safe contiguous windows.


In [ ]:
from pathlib import Path
import base64
import io
import json
import os
import shutil
import subprocess
import sys
import zipfile
from kaggle_secrets import UserSecretsClient

PROJECT_DIR = Path("/kaggle/working/Luan-Van-GC-LSTM-GhostNet-CICDDoS2019-v1")
OUTPUT_DIR = PROJECT_DIR / "outputs" / "step4_smoke"
MOUNTED_DATA_DIR = Path("/kaggle/input/datasets/dungnguyen28101991/cicddos2019-parquet")
DOWNLOADED_DATA_DIR = Path("/kaggle/working/cicddos2019-parquet-input")
PROJECT_ARCHIVE_B64 = "UEsDBBQAAAAIAOF7DV1WZNSFGggAAOMRAAAJAAAAUkVBRE1FLm1kzVhdb+M2Fn3XryCwKNACpp1kprPT9snrSbLBZCaDJG13gQUsSrqW2EikSlJ2PL9+zyUlx8l0u/O2+xAglsX7ce655176L+JyJa/v7j/Iy8b68JGCsEasrlby3Tt7d3Zy+kOW3Tfai41tK3IC/4WGRGlNcLZtqRKOemeroQwaB/Hvb1QGvO3iexUFSt8oU2Vlq7zXG12q9HKjPAm7iW9+EUavevjbWfewae1uLm6v5fL2mu3w+1llh6IlGZzqKwtv9BjI+OjJET71rS51aPfCDoF9+NL2FOPans6z7C5Q78UZTDk71I14LcrBOTJ8AklsdUU/ZpkUJR451erPyPSfyw/XnPlG14NLKbC9PEa63igdms3Q5jHEvHcKiZeqXRfIstWG8p9gzwcHqAZHsnfkyW21qcUn5X4fkHKlEeWW3D6a6JTRG/JBbOG/iv7Ywq7RgXyvSpJebQiJNdQpUSpjDfvTnw+vwl5wuhgCgveq61O5VOWFKp31/uDY2Z2oAUTvU4wK0Io8HVnrKkcdnd7i9MbZTng7uBJYarzEgY6fYYRPMzUAkB18MilKuIxgFQS4SABPbRZPSS0CJ8nMqE0HuGMI1CtATFw9cjIeQaZklNPWi7+efBM9vz35ht+OX0trULuOKq2MQFkul1cfhe76IRzwOHrvyts2RXWBmOAf2cATqvHixQ/afFCP8I14USr22kfKl4SI+YkLeoNaJ+hiMWbCg35BtKQeVE2zPzjFOccT3CGu0wal0uUxeFNtPIhMPjLNE54YQP2cEVxk9NxOm8ruuD9VEIbAI5GqnOojY5FV72djqWeMUgw0hT4al8EicxKtKqiFNfVAJtWd2xQdDKzQkAIFIVU2o9tEN4dmB0vuEiGuPolgxTtkqk3CGk9qNGwjqKrHnCjyPVoMusO71EeWR6uytcB9PGNsRfBaRTgKVT7AU7EXRnWpHeD370t59v0bQabqrTZBQGAa8omVHTec506YUOSqJNMsHYjlUEq4D81UPlY464Kfw4x6ShLKQSRbtQfOl6uPM8HyNYOpDm8jaBVglbOeiahpooNKtiMZJiUkF4PjT7JQrTKcxrJS3a+Jg0yVGEtpQZvU7LAyMRHRVxpeYEH3/Uhe3XVD6mDIkkTsZTND3ZhxyXnBfIeh8iGi5CcP3O2B04UAuMAyEIgNqhLaqMr9jDlcah9zwn+qbWdQKZBssSNdN4zKxelMLH9eydub1Sxq5RA1uVPg6OMsE8INhuuMg9RZt0+RGTY+SjEaNoUJ+B0DIOF9S8/7zPaMLFD2hG9J3L0SQ99alrYh9heZrXY26gkLUMWlUKDzt6kxoLw6qfp3GAbvaKOGNoj3qq4BGzoLkh2g/3meBwyWrBpMbephT+bs7Slm4g+ni1KXVWU9T0jZp1bl17Ps/DHK5kFw1VDpOFST9WQVM6HJ+n1o8FxCUV05Z6/iX0BISv5Xgmhi8RDPLLQBKuOXUER8ePY1D0kkvbgelJG/4O/lMJUY6NM8l9vTRbLhFym2ZDcNtnG++QVPrflede34NdhLL9/5csYdn0gAeMksjLPi7OT124TRauIaj2FMYd/ZB4qaGDWhsODjkcb/8RT4j0iyhpytk82vAPR/l/irJD+LgyI94fCnyb36f0tu+m4aIS2ZGiU8ffPyCx5VUPERjXOeH2lU7hxvNdg5xjfXSZL93PSf85nII1AvHvJK+XQgvTEtTfPfvDX5XNxjZkFyBHrfjk6OzowKv04Kn86k9W2Eeeg65faTsVu14xGmqiqOEs+rZjYKSsAaUfFQxJT09MVEmfMaTeNSW1mcNRY7LnZY7RuBEZyowLVIKyKrpovz4sCOhCqP7SzhOJ9oBJ8uLVeQZeuqw5KeVlWeBadvFm8Pg1HGeVsRr1zYvDLemBCYGkUV02jo+hSCYtu8U/Byj8ygo5BXzQtbykbSIxIcV2rs0ZPuHebXUXN/G3ZWxInkv/uS4fHEvN//Ka8Xozr7xdeJ8n9Xzbpctz5065rF0lBYc+1ffwXdUyKQr/SxUKFspMdQEW9eJ4JzyVfXV0A6DRjcnk5PTkYE5hj0YJqUaXLJo4nMO5h/xe5fyWLAphOyv/28en9+nx6hEBv9KD7dnl9c/QOP1M5LRzXX6/b88urmY86LVzL7bNAztTdYxtrsiZhi+evdsxHZDahUATEaerRm2nTiHel4pm4hwcwrz2wc5+YdwQzmM5NvH69hzHFsD9TzagDCYMdAg1aSRQUv1APbmnqjGEzFxOlx1eDVf7QKG1RY+4DAHQIbuO+Y3divFG9D07gWOE4uy5+xBU17w1v8riGTFuPOYgHhQqiCyZx6EXRG5E6jlbP8/fLy8vp8vfx0tb6/eX/OYI4APM8zbfXoFT4OvON91+4Mo575oRi3/aRBPp7hxX0g9p5UA6kahsYeC8h0hz7WjltIWLwyPO8a7in5+zhaLparG6TRoYkQW9yr2Cz1B5mINzGWrQytVlLUhbQAx/3eFnwhpelCt6iONvdJOVBdHZ4ETJsNMB/M4eiWGl3CbLC9bW29n4uLoW1Fpx+pkuk+CO0i1aVLUBTJo98PtM96pRmIuJMypKOsHOSktRY27HhxwHyqNeeRloN0bWCAWUhBrypjQ/wbQKJPx6jE1S8GAPsbjm/iEEykvTRVbfxNYIwh06wFTFk1Bnv8O0O8LyHwvLDMxGr9dJdbJxXJf0pE2zCHMzQ+u5aj63yBBwmaKDQ+5yTV0b3ds4Lgms+4oz077SeXikt6bIzjQpoc6jz7N1BLAwQUAAAACADhew1dKIu3I0QAAABJAAAACAAAAHRyYWluLnB5SyvKz1UoLkrWKy5JLTCJLylKzMxTyMwtyC8qUcgFsrm4uDLTFOLj8xJzU+PjFWxtFZTi40ES8fFKVlwKQADiaGhyAQBQSwMEFAAAAAgA4XsNXQ+vjMn+BAAAlwwAABAAAABhc3N1bXB0aW9ucy55YW1snVZNbxs3EL33V8ytLSC5spM4iQsfhMRwDKSpGwvooSgIane0y4JLsiRXtvLr+0iuVpEst3VOlrmcD74382ZkCH3norImXHxHNCVVX9Cv8+nd7cebxXQ2O8UhUWU7Zw2beIGfJqqmt30Qjbe9y989y2DNBS1aJicde6otBzI2Us0rZZgq6WLvmbJNIOsp8N89m4ppaXtTS684nGRnqnOyQqTrctV5XiMy3StT2/tAlbchKNNQcFrFr6xp2UeSqxVXkSotAw6klohQ3CIR2Wv4Dbb3FYuV0ixUTU73gVbaWv/D8MXb+/ThJ3o5e3v+YzaWOrI3Mqo1hwv6I6qOQ5SdE410IjASsmZCOFrCKZwlBwWfCbGpnVUmimQlyjP+zF5xP/bwZ4GYTBxILeRIyDE6zg7pWEut6mwrVh6wFbtjjHh21sdAr2ckTU1vZoTTKiEbvVQmIZoAPCRuF2CfnasMNPAo5rSNHkiC5tnJ+YscZnby+uwA/9NdZLsi2wPZafFRiuMY4Cv1wLV4JQbDCRngj5NcDGKX43Nxvf18dazIXX5MBbNO5WITeHYf5SN4r+c3n0gF4gfHJiQ0VijtCNBXyoc4VAQKMbAGmvsQ3m6jjN+Tq4R9ZmzKD/ieHQ5ZTPHQnqkFsLjdHOCaMZxaozfUca2koYOkDzBNuX8bXo+qUAWrhyK0Hsz8lyi4fqlVaLOYyE6ZbJtEIXqGVqCl4z5U7/DmBsYJWc8RD+V6V7e1CtErVO9YpSMoexEuZyezU9SOQIqqk9H6cHk6m02+hs5zZwHzMcD2fAnZRzsh0Fv+RQ6J6vq5iCYWjpVgg4xEu4G9k152jEzCE7A6b9eqBjpQVFl6MHVyAlr6qlURjZq0F/haxO/UF07yGyOwO9Dcm7FiKNdaoE5ugC+a3aPFbZcZwNNbYPd9MtPcIV95BHroTdWGyzPgu5SxakVA4MuzV+cTapMcAhgGI28BonatLEzMa9klmgZADhjYx0OEe2YIbKl2saXiuQy8ny/mxxi4A1JaS/9hsbh9hLxdBvZrFGGZGQiXCh/ooINXinVNEkpIqS5RoEkJ45OzUUGVMQltPfb0lo5FFpLkLWtMpfsaITMPecIpfAQvLBO/GJUYNTkc2O0k5nQ1+h0L9YCk2lsnkkOxc3gM+UFiIbmm79irSoCGwBMM9siN9Vkst8G+mYrrz/PbD0e7wUvXCjQgAO3/bcqhDSpIQQa45mm00/Q3uep6gyRzaWdvpUkGJcr3swlSQUQVNxjcdQO16TVeidaR9V8Ss6faAJr04Nh2j8nKnqkkmVquVh7NB1kpe0kRsOCQBlR/5E2z9EnQlpsyOvpse/3u0wFZZXeYaptmhkETwFMFu7sP8yn6atw08kMCrseWAOCQRbpYivXmlqKl90kFB+nFSX5tgjCtL1Q2HHbHauHRmoM3H645smk8N6iN8fND9qH1RoTeYXVDKa25VRU8Reusts3m2VvR1W/HiiUyfnpYFbyeqJQUJnesjPTxbvFLktEKm1w6GxyMO2rYX42242tcYTWbJmHtc8PXB3P+95wGJfnLSmDX7LV0VOWhtouW5gtgynd4XK+CTCL79VDcbVLnySZw1ZebhV2ftuTMfUmG3vw8ZpoaIlVElL7hmFSl7CrAeKSctFzy0QFYnineiOJYvJwMLxcvzrZnp5D32JskFJiKw1NdEoVhgf9fJP8DUEsDBBQAAAAIAOF7DV3OxBzUkwMAABMHAAASAAAAcGFwZXJfYWxpZ25tZW50Lm1klVRdb9w2EHz3r1igrxbOTpvAxT0ZdpIacFzD5xYFiuLAk1YSYYpUyKU/ivvxnSXvrkkDGOnDfYha7s7OzO4PdGtmjmScHfzEXqiLppejoy3JaPy4nvFNW8KPrDeW81pi8MN6Yyw+AW8+5xeWdWcRK9GyXz+MxuJcokGclIejbdM0X32Q/9KISSwIPfuJ2tFE0wpHm8S2aUkb44xvuSPjO7LT4fHRRGu8JDKRKfIcouB0S78lBmKmsEkcH3F0dta0weXJ08XVRXN5GVZvTk5/phRybJlSO/Jk6MnKGLJQH2Jr/VCQlEtJe1Zm1tb3HKPWAOZP1jeTecZ1EIb4Ld2EOOH/30w9G8mRE0mgP0+OT//C2w9WKHjlwnoaYshzwrN7WRZ6EspO6MjZzogNfiGc5IApcm9FapUKhZ9B0Q5HSvoGdzMr1o/nVzf4uWPjarVGy9RjO82OVdtShICkpusNKvXZLb+8MXEHfkuQCmLRKIROjHb5P0AWFPBQkiLIpJSnuVRQhL9mcZajYrtKwdXSH0LUDg+8AFlW+So/lZjIU3jk/dE+S8H2P8rf8wRrgIwQO5iqkHg4Q+6+ty2yCRwHrdVjq/M/EHMLgOofas2sai5663gRwxPoACyvJlzulURNTA3wbxhKMpTzHSLbyLXd7WsIP17cII+ZRzURy1OIDwSNrFiugNowTdlDgHIlcuUwjXZWUlfCM/1IOSG61m1cgFjkQ4ejPoaJvJk4zUbHZvXLefPm7TtaFfsvLiGD9TXz1S2NJo27qgEydzZyq7pwNzDpejhMVe/C0xI1KPvDmYQ5uDC8kE3wDUiqE/k96uwY29L16v6TUtxy0o5kHwD0DywLLUuJP2cG/1+033FvspMyc6fvQJlP3GaxEHA36JAulZGihPXUMZ0dk+dH9NTGUIfI7GIblXkw87HmETvkkFOV+hg2ojQ7K6/3dccJUjZvT6qTiofBxWiHsXEo6vZLgvhZyngVl1zaZDYOgRllHfBMbDyQYTZJzCY7E3XVqTX3Q6xS4c7uAQnNXAyqxPsgaw0PXW53e+vCASk5s2Gn7NWd77BrE/1+fvP+njAJYBquFx5CtIXjwyzoXsXqPAj+1UKtSZc7UiOr6fTGpJrMtdC+7mvUXTfnd9elLzAzdwGMgxkWrixNGIzh395V5mjlRYlknyqP759blzt1qbr/8RT22F1ftEqA7ffTNNt5v9G+YesfUEsDBBQAAAAIAOF7DV3BZoi3TwAAAFUAAAAQAAAAcmVxdWlyZW1lbnRzLnR4dBXKMQqAMAwF0P3fxckTuCu4OqZVMNo2oUmR3l5dH6+0rB1KZSeDdqpVHqx9m5YZFvlmH9JBteCSkDjApcbzi36YI5NrEv9drTknBHEZ8QJQSwMEFAAAAAgA4XsNXSZNXZPRAgAA+AcAABcAAAB0cmFjZWFiaWxpdHlfbWF0cml4LmNzdq1Vy3ITMRC8+yt0phTb2AlxihMkBeVDSAqq4Lil7M6uh2glZUayMV/PSJsHSXzB+GTvQ9PdM9O9BHcJCXpwscJG09Olxj7Y8s9E9E5H4Fh5qmCNDbgaNMuDxKPZXH+FOhHjGtS1obsEUTXItV8DbZVxjeJ6Bb1RgXyLVg5SPWlMNOOwLVV5Umo/3HrEhabiCKGajWbH+ooaIGiUNTdgj2pvU+9UAxHqzE5tMK6U6W+wSxi3qjVoE+0FdaKXoi9iu1URe3nb9EEtL1gtr1XwFDnriF4YFG0WzK3pQA2MeB/Ed/rc98EQKFPHZOxDv4qmYAKQqq3B/lnt/Fvdd3T8k6UFlFwmrAwJeam0E+tUX6UoBSMZdKolU9rHajo+nRY90/FiOiCjHCQndNbGYjMsQWbAwWLkF/qebu5CXehrglZgO/IplMod5TnlHVEsZIQy1v+Fcaa/RXNjQbHJT2Wbix7og+CUo4MiHjbm2aTe74c5P9EXSV4Q4nAP5FIPhHWBRteiyypXcmXRdQUyEMjYamCWOy/QXj3bBfpOf/6w/KJaESRTHwZZXHaLIYhDNitwynnVYylzGNBTvWRvy4DUJ09ybkA+8s7KGFO0KJOU5PCyLYeBXOhLdJfmV1H6F1jWWlsM4pba550/kMbjhb7w0rgo6XFDw1A7MmFV7CASByMoLPEQt5PsNl17mXLHkzfjremtLnat5L3O5erjvnkNNR8dn4nlpUiXfOLH0GTZJGDVerGnbJOn197bkSyFYsUS3DmVd6/qfHQy1ecEWdIGXeM3rEov0bHIUewT1XBEfiPJ+shrME2x7OCWHVD/TuSt/pjQNvKJIHGi7Cu4Jnix5otmSx8ZOU9XkdlI+h6Qw0x/F5dKwv8G8veBMXnKpseMkBUjzyzjcCxTGcLgcDTm+oekYP529LKgzNKMhzNly4eG3IM/xPqAn0vMK+79LeyF/QdQSwMEFAAAAAgA4XsNXeZDuFGuBAAA4AkAABEAAABjb25maWdzL2Jhc2UueWFtbHVVTW/jNhC9+1cIe7ZTy7Ed27ciQbcLbIsFkraHoiAociSzoUiVpJJ1f33fULKtNLsBAtvzxeGb94Zd8H+TSodZUTjZ0qH43Eu3+B3/H+8Xnx+ffll8PPqYfqW0uP90//DgH1fLcr94KZEQifShWK/wtfUauZWMNJtpmSTXe5ZNY0nwz0jpUOjeNa7pT+RWuxJF9uUPyiitfcwlOxn+6SkhccwQ2gRxNA6pkzgxxokuUKTwQpqPl87UFJOIfYWsw8UQ4TwnnH1cHmYrK7JCSacNLBQPxZ+f2TQfPPPi3soY54UaPpIMDaV58ZQ//0KBZFqcINvubZGns3l+jeDwLvgXctIpEsrbvnUcLJDbpT4wTKd5IUT0fUBEbYCc0RNL8K8wcCGjySVTGwqTQh9+czw/zGP5YV78ZP1r8elhXjzm5OLTl3nxgFaMk8l4l38/TZtz3gnXtxSMEjXJ3NG1+KNpjbUy/Pz09IWjI7IsRdGhBe70UKyW611mBJDmGzbB9x33jPT1cr99AxcGEkkMRcaYzRJ/CKKvHdhIeooD3OVu6vPBNLiJvXa4e+vvU9enq3dfvqmsjtRKcZTxiMK3m/L2dlduN3W5Wu03q+16XW03tF1LUtvNjvR2vbm7XS2rO63X26reyxXtVuW22qr9nVzNZrGzJkXmO44FHilI40QdpGKkGb3lzd1yXixvdkvG7kVapgp8lyDhazFJxghvSgbjQishU6K2S4zEKuOMQFDgBMcItaUXsocihZ7gH+HveTRTRqEgYAF7mt73MXOqsl49z2ZQEwiqKEbjGr4Nwl4oJGFcbZxJJ5G8aM3gPh+jg++E7oGA4i4vgE/d3Pfp2y4cAUK49M5rorcjRB4qz9sJE3SysszwMSr3CEKNpGbYlmW2O8Fcb2XygSHLzCqKQC0UCNR9O07JO3u6lEOdVn4djjpLIEjXUB4hJlje5AHiWGu6Q1FLGzPWPLKhwc6rY2Qx5J+VTOooovmX9bHZZhuvM1RNxN3uhzhpu6PMbd4MBksyOOB8CVyO9zoaDe2LtrfJAHTCMkNT4GCi7vYw1Z8l1yTm93ZqBW0M72mmUIDRZJW7SKpPBtBcV811FsPeG7mEVZjEIGTquPKQQE533lwGebiuHR71de+8D3y7lFim5wjWp+CVFjupcHSjFjamdtHwc+TwHE3fjvwcjey8HtJ5EBMDzmR7NU7716wN2R35bSE10Gb4mt+RYSfpLIysoemOWw9Ar7+n9UPWOJw1ti94X5HW3I827aHYri9Hj1P8v9nKEzFdb/ltwk3fxOUNmq3nML5xxgIPIageoLGjdI5sPFfNTt4THvwbNYe+uc3M0MpcQMAu5eJXUudHT7ySaY4pK76Slp8uBgn7Uz0PCLcERil+blXwoi4nEMbWP4MZZ0m880zVkfsFOUm2Ih77usaqqvABiM/c5SVf7lfXOMIiFe8fok3JMcNg3oXkWw1inNV9VniO5CnxLuFB8nTPXQ+bY9rpUN530AB+Q38/atn+MfuuZgcEhSYlT4M1mzFxbYh1gFUinA/toGTcDuhqaA2llQedx80kvlF/rNWarwAV+1uZmB8TJ1Sv5Sjg/wBQSwMEFAAAAAgA4XsNXYgBvYHVAAAAhwEAABsAAABjb25maWdzL3BhcGVyX2ZhaXRoZnVsLnlhbWxdkFFuwzAMQ/99ihxhwLCfXMbQbDrRlsiGpKDt7WcH7Yr2TzApPdJN6w+Sz2Ga9poxT40aNBZiX8uxhdAUTWuCGcty2vgco7mSY7nN00IsIRSQH4qIaxeSc5XhNrpGCH1vyPPkeqC/KUzgXx9PodBmL4p5P2aPNCwFqsixQfJg9zUahCfVsOEfmgql+oYNi1Jbh/oeR3rvmFFY+DwwrWRrh/Ukl6q//Y6z38ZiXl6Nqe77IZzOLPHSv4wlOu/os+R6Gf0fRe7JzdHiZzzDxJ2EC8xD+ANQSwMEFAAAAAgA4XsNXTSVDU6xAAAASQEAAB8AAABjb25maWdzL3ByYWN0aWNhbF9iYXNlbGluZS55YW1sdY/BTgUhDEX3fAWfYKJu+BnSB3fGGqaQts88/16G6Cw07hrOvad0aH9H8RRiPHpFikOpOBdq+UaGxoIQhmJoLzBj2VeU15jNlRz7Z4oHKpOEsIH8rsh4+BJ1OfNGjwyhW0NNcaNmmI8KE/jr029yOeZ6XIqyUel/orvSeDvxBVzvp9x8OizFyjodqLNZR2fxvCr5gxpXOgHLPAMjP4cwv8zyfeJ/wpV9yUbHmDj/VMIXUEsDBBQAAAAIAOF7DV3q/7diRQAAAEUAAAAPAAAAc3JjL19faW5pdF9fLnB5U1JScnfW9QkO8dV1z8gvLvFLLVFw9nTWdXHJDzYyMLRUKEotKMpPKU3OTMpJBXKKUxOLkjMUCjILUnMy81L1lJSUuLgAUEsDBBQAAAAIAOF7DV2KBfdF4gUAAIwRAAANAAAAc3JjL2NvbmZpZy5weZVY3W/bNhB/91/BaS8S4Ghul62FUQ/owwYMW7s+DAMGwxAY6SSzlUiNpJJoWf73HY/6tp2kfpFE3vfd73h0rlXFkiRvbKMhSZioaqUt41Iqy61Q0qxW3Vqq6rZ/P3JzLMVN//nZKLnKnaiaW7fRy/mEn37DtrWQRb/+Xrar1SqDnGUAdVKBLiC84Qa2LBOp3Rur147osGbqFrQW2clOxK5+WixtVwx/GkxTWrYjg2Mn372Q9IgIcqXZF2jX7JaXDTAhBx2xsFCZMPKC3E/kTBghjeUyhZAY1qQ1whhl0z2vNi7Ahig86qhGSaNpe9w/oH0T1ycbnVnRwAglhuUJMXM3J8waMKmyo+7CXSqeJamSuSgoIolL2JZhCNl/lK01q1R2uoyPj0oCqnOPi7G3up3Ezue65VVJa3CfQm3Zr7T8s9aYBm7c6paxb1mteVHxLZMKPcJ8sCsMUA0yA5m2GGgkAAPSMiXZb7woSviu1uozpJZRDspyUKy5MDDVEwaf2r/ff/jdidHwTyM0ZMwqigbrpfioNJrKPogYlS1aR2JdrNB550tseA6JYw1dZMYwRrEGjK6Fexui0SrDgt8Fjc2v3gZRxNDdh8dVV1RDkJ1JiDUK6xi6viAvqBy4X6xy4sMCcSPCfNVgAYmMW5hWyaye3EJXTUta/ziL1NG/IQE7tg+64AdrFqAg7p6mLoU17g0zjvspGERZERyIuxL05Zix/nssOxAPcjG8bslFFZe9TYch8J5/uyiWvxxsfK3kwYdOh2dlBg10nXDLHjrux6DrJJr7LYKhI9735h/2gWos6MSiApkMlMFgicHqhCzMMa02vI/IlXtn8UAbsW/QzU38Zs028dvDEzaf18WqxlBdW1zGUsblskVRbzbUvFDmJpglHZkGbnTJm3bq2BnqROXJxIbgEPVuujRs4g17d1bJO/Yq3jzl2PO6vJc31MjDzfpV1PlUaNXUiVZ3LjtCjo5QoaEbBgsGQQPJSDmxe8L+bseetPGMoMGoWhlhxS0E49mTCygzsnaQGZT8BsokxbQQoLD8xz0rKsAGV9UX9hEjtyDdIYQ4LJtKznYR2NIK1KnP7UoMp2ywG4g0yYHTFDAnmxxhdLZMa91Hks48cmrN9ofx4OqSvzw/zZqVwljqTly24TmatTt6PCSGQ9ozL0/UU/Q6o+IHsudxSAMnnUzlzHt3JXkFTgmi2XS5wS8MbdFOHFx0IPI06JpA0tMHQ80MEiZNnaA2bHRN6SGoIBNcui5XOLw8PlVfS4WDV16ICyQJGewIjIX6+2Bsf6N02pm2K6I8zAYexAotTyBSgizsEeGBiH39TAq8zHjBO6bCshI4vr8OoufUugRlQGpnGHyRXs98CYlDyhwP5dVyPBNtopsSaVzrRVQam3j4QR08p/4Pid3Vy5uI2s2EuMpIG60Rk464qWs6BS7a1Kce56BaYYSSWpUibTv7Mq3q5E7ITN19jXUXhO4m4r7WzkLz+phkeP6m3fhE9tE3sn2FcQtJu17G8xbNe2tgVKOxCgYnPfBp0sBUCOkPlTPbw5K75SSuUZiap1gTcy9O29sYDzIj6roYmuUIfW3TziF2tVmHC4kXGpoXe9rRsHdfQVXbtutjC/xfX8T/9QL/1xP8L4KII8BdAtUNZG60xLRU1LIoR0eR4dkyXyt5C5qGt9LYakZy4qr/ecKRrzgqxEutRcV1m6RHvIriDWjcofn8TC769nHdhfhcy3giwNcnAT7fL4Zxph+PvE6CIg4lvkPOZprzFe91dlzTEWY/GWE6nTNF5yY9p7U/Wfpx8fFlFpwTN5hDoyIWxGRS7EzyzAT99AjpFw+YCrAQ0w78FU+1SvJXLwf/dXwia9dLcfCfgt5fQHwVE1CfunzglzfC4KCDA+W/dPtw/1rEWVPVpuNd00ye4OXB7P7UNIVAzbHglDa7MFi7ItwGCGuQxg1K3KRC7H7h5eKG1P07Epsjf/3Dj+GoNKY7GoTDDS0+wn0mCuxHYbT6H1BLAwQUAAAACADhew1dsmaYZ7ISAAAkSAAACwAAAHNyYy9kYXRhLnB57Rxrc9vG8bt+xRWdzgAJCEmuk3HZMFNXtjOeuoonTvOF5WAg4kghAgEYB9hiVP337u69AZBS3bTTRzSJBdzt7e3t7fsO2rT1jqXppu/6lqcpK3ZN3XYsq6q6y7qirsTJiW5rt03WCq7frzNxXRZX+vVHUVf6eZd11ycbRL2uy5KvCZHGfVH3Vcdb2Z9nXbYuMyG46TdNEqIBXDCN7n1rUHf7pqi2uv15tY/Za8CbXZVcPXV1G7N3/H3PqzU366j6XbNnmWBVo5uarMqhAf5r8pOTrt3PTxj86N591rb1xwRWD6g6Ant/wm/XvOnYa4J5CQDtnLFfs6bNtrtszqoa1v6Bt2zG+C1v14XgObvasz9l223JTwtgwbYlDjNefSjautrxqqNpm/dswS7rCkimhSbrutoUZqXyLUX2x6ysszyVLScnJ2+e//Hlm/Ti+eWL1y+ef//yHeAJgzfZFS+DmAWlfrhA7uLDWj90sLm8w6fv5VN08va7b394efn88uJlevHtm7/8+VJiS9N11pCw5NkeB6SpqPt2zdNNUfK0yL02YBs2RUBbzjdsnVV1VayzEkgu+12VVtmOh/jPnImujdjsa/wtud9ymKbCd4KIEngqmnCEq/iJK3QiVL/nZteXMGhFeMtCdPQmsZvhsKrlNF3yOWKbumXymRWVehIriQVlWQAKJdShwRSp/hLmJelfMAG7x3NaDOHEh1iikIgRV1J0fCfCiBUb1fU1O5fIqEXjk6sgPmUgW+yHrOw5iWG4CS6IxhnN5JCQbYBE9vEaphBNtuYgpO0OGUhyOGd3FvY+iNxNMMuaYv6mRW7Rv3NQoOQFKPArfCO+uw1j1k/uIm6VxJeopkhTI/qyg2G6s9mHbo8GdxG7y5BQag0578AypX1VgKiouQ8IUIzo8gIsEx93NX3b1GJKgt2FCt6Fx0RWrXADW54rkZQTSvEzb0Xl0EIiMeiSM6y0xJS8CglpxH61YOdHxeblbQMcATvFb7N1V+4Z2CB2p9Z3r3WAbNId7ZGlJLqPFe139GsgP9S2PFsp1osOrXQqsl1DNkMck553vC24kvddIQQafWSQosZTzvBx5kipEg4CP4cDaX7DNDXNcR0jH8nWLUfem6X83tDYtGD/qwwEhRmpulOdmj1NtkcLjpbVzEWkLMcLWSWZAJ/HwwANYbUNogS6yioLg68U2q8VWvz5nAXzwHkbolW8sFhfV92XTwHpo2fxdlitJNllTVhmu6s8Yx+QXXMdJyTiOnvyxZchtSagP3UOk/TdZvYsiKLkmt/mxZaDUEXRQErW13yXkb87qJ45kjxp9o0yWlZjpJLkEAUoS/MTeBWFWGOKohjcskA3l4l1USxeZaUAYy04BAEYV4hFGMQoW/Mg8vgwWK1my7H1wnL/YIIe0IT6J14tvm97Hp1QE0OVAPuBoY/Sg7auu7mMhfAVR6eDtl1WFRvAP2xXUQyJFXCs60Ful9gbsyRJtIamRrXTXM5OeEQoeNaurx2kMbuGYMZaP/KzEiu0xwSzsk7XWNFpONTslU8/ebEFCzQdot/tsnaf4DYGymi2TLWiLjskJu22rK9CD1dk1RpUXWMDtkD8ldBkYCgDPUQEFlx69Korqp5b0wDToH338chfBshsEEDSr1O5HqugYl23uMwz01LWH8FbLygAwjFRQi1h5JKPvNftuHZ69CmWmD8H43925g41NCX8FvYCYo4D477whoWafsshfAO1xp2F2CFtOcapcn+ih5A/scitdCRZ0/AqD0MCi4llvpqpWMoOidkN3y+U5cEYas5C/AVOJyYnSC/nK9yZjvS75RCeC640TcUEhaCoXQt9SEzKi5bkm/1NSbwWRehIjfSDxVD+U+sW9MD+nT8jvRgrsSOnAIYdZjoT8KGLceV5zE3pml7BjJd19wr9rPRQziiDzW0cb+Gkgk1toBZ5i4y6OJjI+cRuAuhjzMmYrZ6kk7O19sOTpkM88IDwZxNc1rhX/RpTmFkDASFvP6C3VnOzj0V3bWyPOJ3iCYM0qv7I7hza7wNvqmjIKks4yCNIoW+1J42CZ78dCLtnY3MOYGRRQ6UeVsWlHQw+00ksuCAjFagdHhoKFYH/oS/QI6H7DtKTYscneO3Eknc+knsQdJlGyziEScLBdN0Zam0sOSbN4bRnEFz1Cq04KcnymBn7TNO6D4FcnuL+hlhykM6NNDcv1pQ5xlhkWHnJKYUS6OAFDUoIR8dvgXXo8EGwFsblq2mUobSKYDaUUIDZcJeiViEz/PmAlIPUaWe4GCxKJL5UHVR6wmJN+mMQHXIAkiCK4R6DRiXgcoCLotg4BCVbsM0QoWZdLwIS16DBIlIeMIgFhnAcBTSlbFoCnx2V5E2gdgCdPfoI5IDDDNBkskY4I4T0tkdH9UD7+gaV8c5ME7jBvKQEHRTomGL8cgJiBV4KQRwRWAZGnRx4AaBRbCer22JbVLacYSb0lJRmJz4vD4w4TsBgDBJhddOlpu+avvsHaJmAf4ASd8RBOpxEAufHuEpP6XbhVNh3gOsepEJ/L8PQrCjBq9DGy5oW5TrCKfaoBszUSUbcao/sA/+AEqpezlf3Wvg19keKrlYsClhbSOcxL91l3Ros253GpQV2uJOHCjMHpMVwHBfaQgpWtGD4YZHjGqIXextQlYRr9DZz1hOkMiAE/V5Nef4BD77TeEcpuEnOqYiheQQc0aRojlCtFNgwWSQaLt8z0ktZZ7URD3DHFF8lcqQVbNeu+cQJzPjhJKbDr764ZkhaG5B/9eTohzVkwdyxdyMNwsH04PTYCupINuYj8XLGKV5RO0DSq9PtrFSDmCateiphhdC9a4urHpUUaxvbtu6btAD+rLkIu7rLSorIgZdc4FbTm01Z4UXJFkgmgbOvfE8hmbl0Kr+wf7uiksgxyLgNzw36yXwFUwo0Yy2GN+FtJIvLtyj8VZOURUV12fAsVhTM2LmqEEe2LCJLTW4M40QsseoXarGCuysdF2Nhsc17Vgg6cBjaFudwIwzUMQjCGsXtavLorLvm6mTDhHcqsDCFLkstcK15nyi4V5p8CaeYJ3dPKP66g5Oq39n9FYbreOSUrHlRhmr9EEw8+eJLHeZ60oAJyXFxOTqjT6NSaSxVIeKlN5TCHTMWtj7nt3LP6RH33ZvYVCDRIkqcIwF091AVA6i2h2zN8axonXXhkgYnXZ3Ks61QzkqtOKvEDTaj2FaQYqdEjkqFFQlUOZaF2K+NUPlVSlOJl91htVBwkF3DtPUuxRCNL1AIowR1QE4URglmXvotb+vGmXt45jA4Y9B6QEdfcr7c5OtK2MZxtAxDJXVpAxk+7o/UEZWMKz2JT0hTZGXK5XU8CLWVtSCadD3LhXdqWS1f122ugQZoLJgJMkyZ0xxasb+RguqDQe1siX8xLRh3lYOo4rknV9G1n+eMN++QKZGx1pBbkbQm7HM5rQ2ybNQwcXLjOvzhAsd2Z4oNGI3IJwPGS1PBFxgujTj3cJDwjlanM5O82Gwg0odIANd/79TRVd3cFPiDFdXkJg8wBqNMHWvQqcRB9955xAbI6oAqSjqnLMEPf+BpV6ttleUw6GhK9BbBX/+KlejTwAl2CZNWDjBA6IatQk/BYa1/DvnglF0UKRh4SMpg9zU6i+JeH3XuroqK554Zkmw4ZmX0MI/DSQ7aB6qPLjNKsmrvVp9MHwoGyukjUECwE0bOBkwKhDSOBoc9OT3VuCj7o4ANjcWdpUQLjIm6vdxvV+e4o3o/AjeiGuhYMD+ges4Q2AUNhnbVDZcwaEgntl2zyAWWieNcS6ObyWgLrEbZ+okfglAtfNvXvfBNiDHCKiRxbe+0ySXeFVWxQ1fbV0S625Xdyi4jkLq4+lSZ63FgEwSQB0BogvvVIm7RFWtmSWaipFgAC36QPwI+SBU6zA0gJYOwB6KvLP8R9Kta7xNA9u+MlmAaRwoiCkQZmfsuHLIpGoapw6NKLWMMPPKIyWzXg7RecdbUokAr83PEa4aWY2GUAdIBHAyk9Y32Oop9bpyeTjNCZ+P/nojvCvYrfd+DxuFRwi5D24bHNXnxAfQ9dCk2QSqqojetDk8fE0Sgv5e7BDmZXZ6MIq3n9/FbuSBSAZulmxx5F2qc7Cu7Ds9rS2BfyqTFHxyFSQO6YMdCYIfuyBl4w5XsIIsID3CNJvZIIcBHUKKFCPx0S1ka8P9M8l8ih8wKcVnk8jAWQAcHuJvgDu3U/ZwCAzofhGdnFffB+HDXnOxapfxoaMETOyw/pFd78ByhhF3On2ECf1Vsg4j9hoX+Aj7XV4DwBwLibYWVKIUvtMgPagb7bLrDIDWMgi3wJoh9Xh6KcCaCdZmBkJUNFSpiuZuU2EMITHiUFnxSwvNApGFsMKjTo+6IqEPoB+6b2GLXJ10oGUQgF45zktEpeI9/5DqJvm0jczJMt2QdMdR0xuwGOLMIZPwaHEzDnAzLcfJC3a74WbMtyeZ/wvX/P2VqRyMu/dOYmMtYzuMhpQHBcFKnd+PukfWYAJlw3rYW/9+dMI6185fc8fG5oxeGe0aFwvBf8slPyCenTPPPk2MOVV0NOWIBgrHqDyc8ahz+E9Laq74oZSkxBW9rDas5pNZnacODeNff+ZdY48Mu8uDpvXsmQrHrYPalf2oiPdLwpGR64Og8RQ4mZwTZsfYzieD41UYqL0NCLrMu+5wvllWDqdEVb9V5X1FtQCK6fWruwVsplRPIDYVxIPjVJlQTqWBqhaEgfY4R0kwLgNtA0NF9+TRmVSaDF2yEsCcaKZ0fiSnMiWv47w8RecP36nCWsMCrOppFTAP40RHtvRsZDpHiagkqmsCsmFuIKtM25AB2+tAmT92DQpAq4Bae9WGr0t7lcEMhiaDbDmIB1oFDPBvg2eF+U7Sic8PhQtzsHXe5HGzb8Fq1b/P/UmEaBBbrDH3HK7wM9foFPr6TV3pev3Ve3kIYiq8vgM6ikjYPAXyUbq8e8batuxqIgOfhMmP22fhs2QYaTjhu2E69K9cOOBbVvZTm+E/lLl0rNXGNQ15SGkdxnnFUqihNmWtSx0qqz2hXy8ABX0Ujk+6aQtniwgzuW1ggE2mNgAkZRmSHAaVJCOTHIqTgdFcCW90vZdRFakfq5UAt7/cPnAHrfR5DobopX4BUeGKEpIDKRY/Sw6U70SqhXoWfUqEqk7e+jY7aCOWRJ9TjRdgu0vBUX8SwC6KKzED7PYvhu2nXCFkkfrszYGDb3Gm9DpfvPLvJtjytW4hhsELaGcOBXtY1JM6oJkPnvy4zWEOebnhGn4xpWXz21AEFme/H9wYM7CElkVcRVkeuLjnqIi/tyECdwiTI0imE+vYKL4BClKliecgvsr67BpwdBc6/Z0WHjVSeqEH1qYr77KnOJZLAv4bwEQZy9xahe21YfQwwR19P3t+mMvIGk74GbOursl3fKt/d5AVaJnwRZM7x0jEobFrfONZdFbDpAqBCgFXuVPSbTXEbqib5ht+JJJ25MWKGJnIldJfR+WBCLSGmHLHqFk+mv5MAVmR92S3wowCEGNyFHEylkxFJmK4+ZH1edNMHvJO3sh1+TfdNX6J8dBXhQFXBuZlDeae8okP3eFc2+KPC8OBuuc1A1HJiv8XF5V6LDlYHAA9cDXSr4uY22OLBi6jepaPIiXD1FZPUyU4OH8YbFhwgepSdOKvTMBCN/wgrQ22nNMFbkgrVsbQ9jt/1emNDO9HiKpd3Ad/KjwRx9NmFVhdn9TTynmqsSXlo6IB9evSg+TEE4LbpJWo0TljjfCoMxs55C9Wuuo5dXwT2bklOGVzvCqQXQ+jbsql3lczicJrVwHv/lpTeI8nHEUs8y/BgYfK/ykqIjje/dbro/f/Ogjy4p/9Ca2J2AZvV3BA0b7vrKQhbjBhVO36xTv8r1okG6gpnqr4WAulyxdScGmE4TCHZpxs2mRRk7RYrAmBu9J/cSC6x7IlXQ/W3rNCIh80G4Hm77fEPSLylnjDnYt0WDa5lETxHc0mXDia/fLp4fTF78aJ+9+Ts/HeHb28CdJLlORJHE4XBbIZAMxDGwAZ8wemN/gMXIA3Hh0uJOYTgY93eAHmnb/qsmv0A/39zMXvz7vs/z765rkV3ybsZEK7pnn04P5XoxCl5h+MzS/kKYnN+5wTPB4ZgzVWPOwqoLNAMzMGMzEHMqLZlvmc7NA5N0SNh5Z/rmKmMc/AdNo1wJUkJFx7th4O0AwHkgb2FtqbSscrwFmK/+lskMY1MkCmeeS82smNohnUK5Z/d+G5wynhLER/j8yfDS4lHJxhZfIMX3jxckrO6hvooqsfX/bGgNsYmfXnqGIEUyZjMcmi0ceqSNGv8Y+byvGkxRXZzNI3/WI5G18lh1Sl9E50CKQs8BEchSVP1zbOUmJO/A1BLAwQUAAAACADhew1dYlvqAIgNAAAMMgAAFgAAAHNyYy9ncmFwaF9zZXF1ZW5jZXMucHm1Wutv48YR/+6/YssPBYmjVfvSBoUSBj00DxRID0Ev6RdVINbiSmYtkQwfZ+tc/e+dmX1zV/K5DyM5kdydx87OzvxmyG3fHlhZbqdx6kVZsvrQtf3IeNO0Ix/rthmurtSzez7c7+s7ffvPoW2utkhe8ZFv9nwYxKDpzSM5o+MjkurRn+BWDozHrm52+vm75mikNdOhOzI+sKbTjzreVPAA/uuqK0m/QEGa/LGvR1FavRZdL7q+3YhhcIT8ZB6K6kO3r0dY4dWHn378y8/l+3d//e4DK1iajD2vmyRnyUe+rysyBN6NYhiTDOb/ySwwBVGfRFP83E8iu6JH7N2+3jWK+/KKwV/fPg5LUHvxLdB93/ODoMdPS1jeAlbV9/xIT47ek5cEfRC/TqLZiB963t3/LJqh7QcpcCDZbBh7easmlqFEM3QMhkbe78QYGRj4oduLsq6GcKidemAGC44O7/p26qIjTVuJEn1MnBl7rJsKuHZjH4yLagfaNJUIl0dDL5GO9UEEIwcxcrT+klX1ZlyBKXP00DXsSiW2jOMml53jTSUZPSXiLe6xv+M5DZjZy4gn5rO9y68ydv1NxJ/qrZwFlhlZ3TDHf…5653 tokens truncated…m4wqVzU3DLc8pkGvAuUZDPn+wy6AZhfVgHDhC8WIt/aVwtp+7L3ioMkDTeLkLweB3oZru4xgPffSEXv1XkqfvssgzY4+gsRTTbuIEJjln0XAlqNPJr2woUb5dUxJZjosO79kJGoZnjna7Sv/6gC/at9ZltHaCx8Uj0lGSgboU5yT737vRxTGLD42xTvItmsq3AS2piukBcDzqz3knlE3aMBho5No9RXGmxECALINNsiijWFzacMD1zA6fJ3u+RMSbfIUXyXS/lUi8j0ktJ8Ue/0l+gnZg9Tr/CMPqN9/+dnBckdLgzGMGkn1WTIlA5PciBWgx2DllBRmJupmpn0ZMCRRPyTtb3pT+gEoBBM8bT/paBtCGLbAt4TOun30I4nu6pqpR9gPHrfi3aTI/P+jAZMHOtpnc7af4AFO2hTJPBCF1cVX/yys5iOBe5vJhlbHfQeYPHYy2fgzqHBhG5JNvmZxfeLAbPI1EvVxk/U/wivwzoWAwdyNvw9cEX/geWdrlobL4NE5ifzANukpbK2ORm1+ynFl6dx2f0NfEG6FFSvquxYIUsOqg7yHSCYyggdq4A2+fF1Z7tvYywT6nZWQdhcjPGqd7+OY4/qxhzoB4GATQnune5jxn315wmPjzqro6haqmttNwB0FXGz2GhrCRmNHy3QuRPYnqzRf4qwPkEEtelBKMsAoSnHIpdRjqnsGWPwPUEsDBBQAAAAIAOF7DV2Qps7XfhMAAK1FAAAPAAAAc3JjL3RyYWluaW5nLnB5xTvbkuO4re/9FSqdF2nGrenu2d2TuKKtTCZz9mWTbO1Och4cl4qWaVtp3aJLX3bO/PsBQFC8SO7unU0qrqkxTYIgCAIgCKAPXVMFWXYYh7GTWRYUVdt0QyDquhnEUDR1f3HBfXl/p5v/6JtatysxnHS76XWrP41DUepfQ1FJ3b4XXV3Ux/7igEvvxSDyUvS97Ke1+32RDyszpCBbWKcsdhrqB1yWBobHFvDp/nf14yr4Sf5zlHUuJ9rrsWofAXNQtxNNTZczhv62lEBVUsmhK/KJkOgigI/I87ET+WPW500nV9SXN/Vh7IE7Gey+Kx5Ub9vJvKBeaIiyzA40JevHFvEpoK7JMzHmGlvMe0Bi9Lp1bXUmyMc+QWbo8T9C+/tG7GW3onYvhws1wwG774pBZnRUavDYifaU9cybaZuaWd/h8EdZ903HLE+qZi9LDffd++9/+vin705NP/xZwvkQ/B/EkJ9WAQFmregE8FB2Wd6MNRB1QeenIPUyTHHE3/Ga2LKXBxDDoi6GLIt6WR5WwW6s96VcL9IXB5ffBn9uaqlm4wcnJWpOkPLkCwt3KWtGTZOLejBzOwniXwcAElloEs2q7CG2MR3lAJytJkKLei8f1oiRMKP0bvoBDgdkcWsWkfsjyMIgQBJ0u2mBVHtF6r8HhM191g7dhlAHa7VE8Dq42U7oamC5RsftGTrqfxk6ZsGnqQM/oeFAuA6WWaOQblfuRKAL2JQ9etN09/Ik2j2NeNPMwGa9shi5nvjoo6KdkxSGdDKRYdGlxbrYTPsM4ornmzdlKUBxPG2J8Mh7I40b95i3dPRGJdSxM1NNd2SJq2ZgqvQcNS4jOxXVbQLU5bfRBhfd2KewDQ5NF2A37Iq++20cJ4eyEUNk7UYzmpGLPhtIczTK6XyWEILtBaMqeW7Z1EcLszmLdBjbUkYz8tUK1mFuY0ISxfO1LMTmyPpzVFun+ovoBvW9+P10n0RA68+yTj92o4zZRH24E+VIF96Psh9Ltg1l08ORE3Odu8Duq0TeNdlk++dD6jKY9x+u7b57WRxPg9x73XhVwJXBXcH/kdGjkaI+yI6kopdwH+0dQs0gynINt1fVZy0Y5l5ULRpVAzqcumY8ntpx4EEGJKQ24GM2AMPWQVn0A+jvsOVe2Pre7227Zid2RVkMhex5kP4jdNst6xrehJHGC1Jf70XXCbjAvfn2EJ2XEgJjc23urPVl3Mt6AIMIk8e6AA3ilWK17e7RmObiQDbCQh0HaRrcrB2jAlB4QzDmOPidD2Cp/HRMXr9SVMcJYLK8TaOhuwbVvji/uk/yemlBh5BfSgRc7KALRUarpGFz14WrQNzJThxlGpIYh0yifMhlOwR/AzWSH7qu6Wa3K1Gizl0qbZPKHJLzAIdcJ39q9mPJPlZJHs7a9naU74VuTUea5s3Yy7siB0FS6q9+8RxPZsDvQqlZVnoiJ0ESIxYVcIXLDG0BSNNVcmV1ssZgP0vVOJy0wBup3QLARmuGxJsDPeunweYKtAhI15jcQwc62Qmo7kFtVHZM/n0xnJgnxizgJiNLYtCY7vCGQmvKrHfESQ2m6jsZmkjxN3aAyuZYDCA1GQASGyMC94GIkdM5RnoWo+abKZ4vn/XFz+jcoR48BWwd2OuUpR1/gUzAFXCK4iRvxyiOg1cW3gUM+nRfp+fgzDE94gkQk/vmMFTiYdrWvqjSa49AEpNEtK2s995WFG2Jukrj2FtuEh492aIgAQy4sFrwaTyWdC1hWpgrS9H25wQNnCoWROuqULYXrpEc7yBUd7Xv2Lo45jDWHmNPFR4zkv6lOdZ+4qVreeVcxqvpClay+szLzXht2koq4me2cBX8LAHtvrgjbOkVex/4f7bCf9Y1/69aWqN8cnW2wb7FM+hRP1JLcd4EKEpApKMKlrumfaFU6Zf7TI5camNrnncwPN3rncEr5jjAqmsGebh2oA7XFoTFfQayeuy9Ka8rtfyT1VwOHV/Emj3zzFLWnSWQRf9MTwADdX11BVfO88fxpBuXuvZMIdNEBdfy8vrGQqU2nKovcMLRp47Q0wKzjxdR5MDi+abq6xlYR0vTGTv1bDoYf37MnsNOlALYtlcuSaaOr4+U8XyJp6hstAojrDmYg08OZVR2RU0/8QkmesKjcevHBYwArm++isGIFDVcRcfhlPquWGz59IwadbOPfEALK+377Y02etp9ZfK+ZQ+DcW4YYMvXIRMZw9nSGjyc9GOlLjqFZprmWIXZC44XiacIzk9v33VDcRD58NfW9g6csI0TirEejLXYlfhE2DVNabp3Y34r4WDgCc2Ouy0p8lA80Jjp7OSRvL6lCSDPYA+GjpwlcvDMrH+ORecufy56xJSilwOwEf+MXSBFN4WYsOEOKsKVWYcGPOO7oo3CN6GHRO0F4FTDHbQ2AxB4mlbPDJHanqZZ//bA8rJQ0uQ8B+BJYe979tqpm0Ef0/ydI4peWu5+FF5e9m8vmTtFPzE+uD/JGgQoGElycIjXC12nhMOMu2Zo3l44I+4WCIB/RmH/Fu49xcWsFpVMVdsK2Kl1OVZXNnBpZBhL1oKEgWTEAIa4uJPZrXykEZIRZKnzSkSOnGcZq9P/iLI3TMa1gGpcJjKrm63DgjAMIpL8oyngfQdeFHnj1ABnPLIEy6UTDr8tRS6j8O9/ByaglMVIJM40+EF/h0zSiyz4QI80lDylQL5A4LpiGGTV0tKdqI8ymsnk6+Dae2s6j2n9waB+NziHB2ePHKSlRb13FMoam6GycCTqOLNDUQJpQxcRN1c2qhXyNJ5LrDodjPo4Y/xyNbwRPfatg+C/lKzBr7bBBEUv0ecqg4bgoCGnOa3Ib8Eb62erGvYDswF+BgAHpln+u5n+z7lK7EYHvC+lbCO4gqKb4NUrjWMV/Mby9SvZ90AVrHwIjQYeRIFWDg/7E3IvQb35vA4+0fVrCI6TjFQqyz6HM5sx2VVXA8gq/AhXDZCo7ALTEAeUUjDYjVPGCaEEGxO4Hz1QSsVho/wk89sW1GUAZXok9VZvlLbJT9YFsBxZwOOr4CXX6VABdSR/0d0KqodFcA5AYU6J+nYSqJ8CdKrvBF5KA/Kv3uheVHjFUQFR1Blnl9YzJ1wBQWexp97nIDH7VBzXXqJhykGBY5UDE4GliEhg2HMZthtr8LyOYwVq1c9BziYzZqmCkPjOgXZqW15fSIcAgyquAs/EQWaI1nYNw+lMAHBqnwOejgbTBLrtAJOY6hHH7EgQI895CPlYQz7XiH/bS6ojxJcRQDnnmWCnBXnHLyiAmx+pDwzvi2U4foG4eD0gmMnNSQvmQPYmWFRhHresMSVUMKQa1siySAHk8oA10xEwmOD89iXEZA97LSx+QjGiXt6Rztf4LOhUKG+mO2fFmSeCkVSZZ8YQ24NJ24Abp15E4eIIvn+WRpxHT+i43Qyn9wHGecj4YKKnrUrQjAO89NbkVXjerEnLI8Ky2F1wPET/TsYeXIZ3xyNTM5uQtI/YwnuwLYcLY1nRHd10zf2GFV7lYKADnQWmWEUlsR9vaLo/6D6Gl2sxlBgRKMUOk8p1YB4MUTiFD+DrTiZtfUSPRvdi+yMqHXkNRsgDG+CdblsiH4WocC5SUs5zCPXg9/jtIEKNdIlTKorNvxkE7/76/vLHv7wngripn7BWtLU4jh3wQjwUyFPgMjzUdsj0PoIxDDOm0X+vgq+Sr90MADmLqVnbvX1R4qU5JLOtcJvA6zBCt+jskekP0pQgKcqSw7uXAs+ixvVpBbSnxpaqLsTKgzXTAWJKR53ahJjd4OT1cyvTPsjchtsNkL8ocdM6CjB+EVKbOc9jXtwBocZSBpLslOX7gad8IBXR8p6qL2/ysSv2kSjbk0ivkpuvvdFSHjEuG3tikwz4OM9K8QhWYD7aizt4KhwjZSKCN7Ye7tsivf7mysxByctB5mWkZuswS6aKRw4FeLlgstXTv1cKa9uel7tAeKWeM812kgYJ1QkPQGJ7UHjLuLlO9nmkuIVboAISsmo3DZFdJDxI6fbfaR114VA6u42sfcc2KDxj9hi5/hROW0OXWe9POxnWlsGguBvFG9jp+MxRp6nux8iAvvoT7Aah5J9PTCDCNbE8S7sUahamlawJ/VhVAungOXl/F8bgWss6Cu9hci3vy6KWaQhtWefNHvaWhuNwuPxNGCM3T4Lqfcy7gBJEGAHr7xL1I1IwsQfDo819tAnV8mh+yQqF27PAfaRJpeoBnedQFV0UeHOLvOyzTHRQ2O1TkXnS9D5Vr2cMzlmSHcfL3PMX+0+xD9R8FOUbTsJQUuGVRb7FTTSXtNOVtpo/F6291RWz0osVzNZkJK/gxxT05XVedE9iLBvuSp/RwavgKvn6awzTAsA35wD0gRSVeiiT5S2q/gSkcXFfkFeiTcM/4KUW2oQleVM23U50Ec1GOlOcr2D09ZA9DEV+20dn5GEVODzruO4y/e2Vh+bx5WjcmZG+kn6wTpWvJfCa8MBDds/S8L0WRD49d8MLV8+5a2cu0spvcu6g+f2DvYuusHZ6HRd6M4nWtC4ygISTGo6biR/fEi4aukVjtjqjpV7l2Zltn/Fw5+6p52dOyNW+deUMP0Epx65CE1Qp5zzjlwonvyBM8FS4QXEdHt2dE061ngxWNMbk0K1OqrxFIjq4Dp1rvZcYtZ8An4lWIMiocxML+Qo34vlEdIMv/LanAmZrlzqIa/YcW6NJdQs9GL5F+qi2DAw1aOCQNbdcaobQKuJUiXrEhBxskdI12GBDBM42qPi+qZKlUfDBFYZ83Iuk6DNxJ4oSY9F2NYcFYq2UibL08akqjql2Qf2MQpwaPrmYcv7DvB3ZQpA9AZlvqRo7PSNim9CBC7e+64evF/VCtC6YDPUYxBdMV+TM11c3PTHTUlS7vaBiwDX9v7nexvYKFEXgjNnMXB6koCJ4DYScYq2a3iBW3WvSn0QrNzd8ValS6dSrko4cpE5KcsWKFfvFNEpekREmwta3ZTHYtVCuUVsssFa0x655MjpIOUvz04PrT+PhAPcBLUxPTsUBF0rXyh7q9EzZrAtfj1V233S3sL30yjdtdAZw5rSkLv/Gg+dD0GfN0R/83yRYz+SFZ+enq2zc3L1/BFNtEqZu6+R9B1b6Qz10TfuIgQHOj6ZTmpTNJCySD+w5FsdNyH9VMXnyGoSlfopzTuqnAtHv9qL6X3PAKm5qomN2KLTsuI7BMaOziodsL3OhKzU0FZvQHgRHWWfa6fyn4KlLW9llJuL6vgHdlu/qWsLzuD5+/6MhunGD6fj5CEr7kHIZwxQp7u3aEDASWVXUM0IrjL3aO/SoFVWbeclbe/IDyIWps4F/ZN62MQV/1LknmPsgOacx5oGwGQCLJKBo+5+om23kSiczU4uG+PlXsSng42AzdFzin9JgjZ5zmcHA1ATtmN9sEaVGVupeC/B9oFpXK5U0maqo0BajYaabLflBdURNj7GhttjrB5D7qCRLTpAJ9wGBTRQnHRUJRddXN19h/ulGl9XqV+uzdYmo6yQCJtXoCoaXaFR6QAtYUY+mxmeRX51pjeRNB0c/TCWazpCy9FeO/fHKII3x+IKCSJPKoNIsMI6YiB6yoclqOCTLL5gU1lRrinFocgHPIYU5o0IRS1yXZW+eMnxRVaYC/ILKTKUlCX1RlWWc7ER+ey86O3RlQY41NbJoYo5fvYmbB7Or/vIpL4uWGAcc66osWjCHgW8vEByTxRlNxnnhNl4me5DtOUI0vSpS4x2sJXbP1Zg+V67qCepr5Xow892iTrRQHipV4uMvu7wAivvr5+tnZxYACXA7Vy+1CgazerKoWAnmwnUNOB3oaqZtKyOHKzbS7qU9IbaSXs9jt+O+L18CIxyp/8dROutJ394r0L2t7JwmSS4IdDO2/eZqCw5xF87+eopYME/v2Sz0HDZ7Y0/mBZdns4g4aU5bxqf6Q1uWnkQypWbWvoA/h+vz1NJvda5QhlOwsiEVSOAd3TCz3SW69jP4Vt+v0zxz3yIV/Gv1BA4rWj5lCs9WIOiPkomApc94Q8axWgXT2rNsrK8uC+Tp58PqXBr2XLbVysAgjRnXJ5l0gZLr7JMqprh6u/+ctIMpAOEid3EnI975ysJk1Z3RXwAnOfjMN5EBWJmVQioGMZzEZZxslz5i90p7AWLiLF8UNk7r1DSffplam0oBfXj/SiX8ojw4fozGLMbyF7atI10LQ4Zd2u/kmiuH2QcLa//mkxlS1UTheSxPnv8qcPB+mYScX9EVDG+xc1JzHt0TjHUwn4Wz1MWqYwGXhOsX/STQ2b++8P1u87Qwa08vmWeYUok2w0pFCkPra5HfuvB6Kh9T612h5lFJnlWA46284SIgjpVYSZMnHQEAe+klrXOUJIKUiHsig4kftX1je22ivJD8PBPn+kLWQxRtrrLLjl0JX1IFo/YWUiBVPyZ+bZEMzbQ2g/U91tYMjBOnwkIoJ25l4Kb4l6onB8hPHJsip3L2d/Mviuh89gutXpIONVNI2OwStJn0cblK7E+yy6X4tneLwGwk2vZvpwKkswbXiIE2B6ZnSV4T+TCgg7N5CsULzI5Om2EZLy+A7HYXW581bBoOrfukMm8+6aZj2edFgKj+Ix5V2GLp7z78zx+RkigrwbPkU1sZ+l8nh8+k5399wdv/A1BLAwQUAAAACADhew1dWd7nQrIDAABJCgAAEgAAAHRlc3RzL3Rlc3RfZGF0YS5weZVWS28cKRC+969AnBip3Z6JpZXX0lyiPBRpFe0h2os1QhhoD0k3EKA99kb571vQdA/z2jiWNR6KenxUfVVl1VvjAvrqja5aZ3pkWdh26gGp8eJvOFZVPlimBfMIfq2YZS9B+lCNxt7xRrDAJmtSIfjhTButOOvUv5Jy0w299vXpTetYL0e5kEHyQAetvg+TSb5Rnpsn6WgM42UYpT6wh05Sz3oLf5TI7p/AMejJSZn2TKsW4ML9oqoqIVsU0ecIdLdVcLSMgw9PS3CC7lTYmiFQrzqpk0WnvDKaLO5SrIQerSEzzTuI9iEeyQ+M/mIPssN36B6/ff/508fPeFMjjL6oHuIC3HTznIQfOrNDn94lCcObn4vDFIHv03SR9DlqMu8l5BxgBTJrNjnfC7Reg98RDQTbAygib0o/52pw6rcunXbpy2b+loLm6+Q6JjEzpnFMeenJP6wb5HvnjKtRzwLfrvGcW49zci+xiBTBc6Y3B4XdM4IyBwQZeQI0pt4MDuoMbLJS/H8RZwiYTmatSl5TrZarq9Wba9Zw/xRxHB9vrlarfNzUZ1w5s8ueljVa1WiZtXL9W+V8AEgnFC9L7yU3WlzWgmrZF7I44Ely3MjvA+s8GR2cuddj/Ukq5U2Z2qmXoBY6OAZU8bIDxnhqneqZe4Fsu4GHAfJunQSfT0o/wnNNIKG3NA6auzRfcvLjDTxhukPXCHPFhTD+zXL1J8gcAAmTLylGSqVxsx6NwSIecfySyjCqzF1f6M2yvZem/yaUIxAHGtyvv7ghZ3fWvajRGoeUFvIZPpFj+lGS1W3BXZJQXqMW56L/SNo/m/wovGiCGfiW5HLyrewZ3TK/BciYPfARJdm/JD81DjU/9DHdTZzh4GjnYIjRIJ8DiZJGDL31JYlLCnMz6ADcW90CVYMJrIt89FGyBDriAgnIilOmaI2k5kZAYdd4CO3VLV6cQZrnMHQ0dTIuhtdjDSwMEQ62kZYidpSMw2JGvpwkUet+81u48tgfn/VqTMapR6UhU9l8QnJTdDdsCjuEY40/Co2Lma0vRkovLHfHfuqVQ7xwYB3sSs00nycmZdZK4N44uigsOZs6VLCX6ORkwNWnk2qeT5eSHLs39trxrp7bvv5VZ8MgvM3rLw8XcHd5m5MUsUZFkeSzhVkkRfkePxL9jNKZRN+c1SvL6o8qug/5itKmBRn/xRDpTWNpTxbpXj9MRT+2OWDDrIUPl0ge6VMy76fw6R14c7ymT9SL6KXJPnZV/QdQSwMEFAAAAAgA4XsNXZBFb/MsBgAAexIAAB0AAAB0ZXN0cy90ZXN0X2dyYXBoX3NlcXVlbmNlcy5wea1YbYvjNhD+nl+hGgp263Xzsl1KqAsHx0GhlIPrtxCMNlay6tqyKsl3m9vuf+9oJNuy83L50OwSr6WZR6OZZ0aj3aumJrtGHgmvZaMMKRmT9n22tzOSmqeKP3aTH+F1NvMvoq1BjWoiZDckqShhAH5lOXMIWu2ykhraQWhay4oVu0YYfmibVheSqn9aZoo9r9igc1BUPhWawZTYMd2pxzMCn3cVPwhWfpIVNymOUDtSSMWkakBcs7LQw+xjy6uyBysctmFCN0o7CU0/s6sCn2EF2EcgVDH6TA8snSWD1YMBXBx6rwVWocl6NpuVbE+0YXJlXbHnhzjAFQfztCZcGJKT+xTEFC9ZN7BMyN1vpOQ7s0bDFDOtEuQVX+wnQthoHQy54fECIDAZSS/Iu/WtPP4xEVMgxhVGVLNda7h1ZNMqUFTNFw1qf6l2qmSoOkDIVVtZ3Kii2hSG18yaHk3tcGBMlLIBF8BCVVsLq/YJZ8jvH6cqJQBxQQ1vxDm998P0GeVe4Ynqp0JQsErSHdrZCm7uDGhPdWqOER8Wk03Fd0erU6pGFl+4KJsvUy1HsxK8t7O2oDS+sDIQfZu5b8cZ6qjv2B3jNwYmJeDtgpd6TSquzQaM2KbkoJpWwiiKkH/Jn41gwCH7QBqFieTohBog0mmSRpF99IoLvWU7/Xm9fqya3XM+jxz9IMYgLsvsPWT5BwXuikMuWj1LnSEZ/dYdPEzhn8GUrxA4t+mXXr+6/b1FZA8muRdIiW7b2wCh6AhoS4oDGm0hOifrcEDUAwYyA8/QosU8sz/eIPI9WZEfyeIWwybEG9AWAdryBrS3BB8v4HkhM6qoOLAYsji24UjID2SVktIcJcthel811KyWSaaYfqIyEEzJygEdPZCmStFjvAmMuWDGAA9Me7hPwloUsgoXSslLSo6Jp7BNoAJLssbasjNDnS/gBPHZUgDNaOVqsY5NLQt7Gq3xEErWvq6LsrKEvlbg497/49yJjKJcRCnmS+xc+EuSgFP6mVGBdlt03xRqOVR2t37Wr/uSoYNJnpMYInDf+/eC/HEi74TBpwaJcsicXoFRKUCLVrHH8BX0mJ6CbtYpuVtsz63MygMkhChZaOkyJYvlLUvDbMn3+zgE87GSRoHjLNnaqnJ7Ga1vyVWdIIxSD1hCX7jOF4m1ajFSt7qWtaIpGdZllHm4R3b2g5agHrkfY3psRwNg4hi7tIusxqB+M9iVbsFvLSUdYUfL9zQmPwUHrFfNhPwaJRkDJxgdX9YbrXer0sTamgq+h0hnf2s4eAL9IEWD6ByohNSEUx78VzxCF3BgZfy/JuFmDixMyRKz5ueUPGyToHB6qX7gWueUr7qWCaiU3py0m/nWcmp5TnRE08w0WDGQgpvObqiIm87ybejFnWq0drsdfGOrnDsCfRdZcF2wWppj51XvN3DrcJr60/42P5/62p3BYbl7wHLnx2/0qa9og1vtx2YOwtisGYIa+a7ZNje2rNrOKfHtjDsvsEHOL3fXsd/XKIJObQONLjWtjjBukbSTZTStGoDcYpmbo5Hu1R5kHUbXbUBZNExp14oBaIaikBInhehbkF1vcwUx4Af6qsBI1Qw6RyrBj3AQNq2Bp4JM1c9QZc0TDBSKNapkCupzxxPff30zq1ZbOP9BFpX66whont5OBg45217ysDXYQOG0ZF+6xwoeYaoOEXd6jtSxtSMJxHDrVwVw6eNoaZCxYheWOw5o85P2ZLLyrbJoBBd9IEbm4L2GfKCVhod7sd+heTUz1N5+89e3USVy0eridube6tumfrRvSiZsRBwM7Oakld2OK9Xc8+CGg77DfbGtBIYZgg7q9ssGfETg7v4+3BcxpVThukENT2jjggu/asWFfg4Pq5yMzi3cUuYXcel96bJxpu/fdAC27d9CY7yYz+dX2/+hn7ai4SH0B31kFWK+m0C9JeDqzhGxNd1dx1wl0Pwry5cgS7DzypExXSNhXQVbvv5vkSEjHbaThlbpAUA1Y2V+D6GtueB1W1v/4tUbpmEQeioc7MzR+f3pmWh7Kwfqmq+HuTvaIXTPtrxAYsRe4BzPsKdLMjC1EjSGBu4728Bluq0n5dMD/pqT1ew/UEsDBBQAAAAIAOF7DV2r+Lex8QEAAMYDAAATAAAAdGVzdHMvdGVzdF9tb2RlbC5weV1TPW/bMBDd/SsOmShAYWo7zVCAHdrBS5ul3QyDOEsnmQhFqiSFxv315Ycs2xEEUbp7fHfv8aSG0boAZhrGM6AHM65UCQXrmtNq1Tk7gHcNb6zpVA9zVltsZQldIYNtSV8Qu+8/fv3+uTtZH14p1LBzOJ6+YUikq5Y6COSDzFtkZ91fdK1E08ojNm/pg1Xw+BVeraEvK4jXXF/clmYPZfVPR/TEzzjohxqW4OiwCapBLVNaKzNDqkxYuhUfGmUdYZgcxQqTCeKlhkaj9/Pntp77EGUpTNT25CNTmEZNLBvHAxlvHdvvP9WwriE+N4ca9vF1k+/14VBB1A0SlAGHpie2qQrfMbmUOlssYzmeLk9/JjINyXdRCiX3ZT4/ZkYeiVo78OgvTjpIZ3q2rbixbkDNvPpHgsXizzW8VFXFu+hlYFW9sAd0PQV5Fvcicvc3sCRYKtPSu8jarxkTTS1e+Q8c0brthaPI1LZXwdeAIWKCsiZKzofCsgEFFL2nPG8Jy/0JRwIhIKnY3iEWlnvQ8x2otIRaN9p6Yjd7poG1ahDrqp5BcfB8PJHUntViTY+fC1PJGsO7yTRpL2reOBtHJFI5G4/hoiur4BdLK36d7Lu+zZmN6HCgQI73DltQ8T+0Ic9+HpElnUYlO8SXkGdxaP4DUEsDBBQAAAAIAOF7DV2r+kT//gQAADUOAAAbAAAAdGVzdHMvdGVzdF9wcmVwcm9jZXNzaW5nLnB51VZdb9s2FH33ryAEDJA2RZXUdmgNuMC2rkOKAgna7skwCEa6stlKJEHSSbwi++27JCVbStQ0r1MMx+L9Pvfwko2WHamkOhDeKaktqQGUe180TqKY3bX8ahBe4uti0b+IfYdmzBChhiXFRI0L+FH1IngwusoqKRq+HZy0ktU0LJ1UlAalZQXGcHHU/Itxcd6pvQWdkg/AvrItfGINXB6VpV4sFpcfL97/+cdn+vHi4jNZ+SRjShveAqVJpsHI9hriJFNMg7BmXWzQqIaGKM0qyyvW9unEyXJB8OnzXY1Tjb3EPZNwz0gU5CZyv6+YgezAujZKn6R/ysBZtlxMrJNRNutoAlG0WUccC2OWS0EbiVVatwaCXbVQRxvM/h1rDXgXGuxei95TX7xFC8prRIQ3HDSW2e47YSiC1P+mgnVAjdUYzzzA5iF4Xs6MAWzdkHTNLPO5PgiEqzlmuSLR38IFqpckj8YuWNvGHGs1lokK4mCWEswnIVgwCQuEiycFS449H6FIG42R44ScvUHGZm/R/p1bCaVqeWOw0PXGv7mQRrXcphhvL/CfbBoD1iUQx5HVSNYoJWWekjxJSRxds5bXvj+4/DIlRR7WHfBhpcSVHtYhAhc13DqXmomtKxojjVTcg373gHk1yE4bB4Nf+mSSiaYrIGNKgajjbxOJeyJfTbTsq3oo/8CuoEV59FtEeNOn9hMpXdNyAkguEv0ezRg2BVr5NGeFtN5jxIpZeEytRKFQGReNC+5z9GwJQBOcNH1Gp2wCMj+TcsYfcsQRydVbZPmMwrtW3pDztyhvIoT25uybj3l39s2HuZsr9JPc6wrI+aVDqcgz91fMKb7FnnPhyTDVLue0P/MO9VmnnGJePCvKZ2VevCJ5vvSfOZvRJloGYGaUKK2YwlEAtGaH4PysmE2BUuNrC2OU1x6WHpGsMteP2iDvgsn38jBY2wO3y0eA3mq5V/f17+Nwl4yH3Xg/x24nJOPBd28McGuoFO2BenZRJJfTwDF0DYaeNjL1ts5ZbDtF3em49OfNE4fj8dRCle+cZ3EwwDkHUK9elENNZt9a77jXyzBnl60wODY66iEx8ex0S4etPJnQwWUWCr7NzI4p6Cdymc8ojlCYar+c8+pwekRtOgX8vAuGHVjmxjgOcy1xctWj4yI6WoQ2RZuJy+MOf6q7wWDW2zAOiJB21mEDzO8kqWvQU+MwmU+tMkgE0Jmzox27pesfOcv8TojdIE02iUOveJ39sCnoGo+yN26+TdmWGYYXoIGx6f1aJrQ4qoXryYhO2ReDR1mSwS03SLWnWGHoL/IKL5Bjs9Eu3Druma9cGXrD7U7uLe144K6f5cdbR3hD/uOJwMLRWL5ISW0PCla45hF/XvrbnuNc/Colz0OGPFwg0XZ0nYxByWpnVkVKrpitdtTwf2CFHncc+aCRYqs8e53iHUTt2KrAM70FpoVLrBfmefFwTp2eHa/xEkI7xJkjaUGv3Kkz3dTH3Qs1ZtfnOd3XcSh8gvWg6HBzjBZyAM1rIRrWnzbbLBhQvEhVrTTIgFPAlAye77cjuDf3GuH7QzXgRNuCAAQBR9VsczQ7xOsjMuvcle1rR6lgYnNCbT1axnZl+VhWOtlz9/ViKhjW8AZ1FGxmqPDj5peT5pf/i+a7aeRuRQYxG/czyZg49HvyaQxY/3v008fZDKSYFWkr21UBZ78mi/8AUEsDBBQAAAAIAOF7DV3uv6dVkgIAAJoGAAAUAAAAdGVzdHMvdGVzdF9zcGxpdHMucHmVVNuK2zAQffdXDIKCTVOv013SNuBCS+lT/2AJQrHHWYEtq5K8F0L+vSNLsZ2S3VKRh1g6c+Z2ZmSne+NAC1ULC/TTdZI0pu/Amiq3upXOggwgYa08KH4w/aB5eFrBo2hlLRyGCy6Vw4OR7iVJkhobsC/KPaCTVTDDmjdGdJhm8OEr+cp/CCd++pttAnRM/2ShhPvd+NX0Bmw/mAqJt8ZnkAqMUAdMN1nA+xMQZNWwSmg3GLyJRsel8Smv7CObrDw3eeOyXrAWC9pzOLnQGlWdHi9e/GGcRweNbMlLzbYxmNUb2OCUoOHPFagVnT7zNSwmcdoeg8GJXTH5JfbYEpx9YyAbSGNmNzewLuD9RREzeAcfoSyhAGwtAvv+F+EpC61AKqW66JGntVnsrEPrQlMtFwZ5UAc1eI9U26gHy5+ke+iHIAxjsXKyV2ks8ygFatyrIomB2KF1BLuiv3Tup7eYE2l9QXjVt0Onylie+ZUCQsOdEVJ5V2NQZZF/mhFR1nQ/AXjf8IUh4dczPgTl61Oui/naItbl3cf5oqJBC/MinMNOO1vexueQ7TRAlPBrs5WGkuRjzsGMaoM0ohPknlkn3GDZzreaaf9esyXUorvgIQvvhe0yb3FkY5JsBWwuhf/ybWenkUhTKca8KdQlUz5e7l9SFqpCQs4m9lwNSv4ezs2d404nurwTz2k2hrFegqIPg34b3TNsGi+nR5w6FLOdJzUmsYUi39zOTVim5N+KzZfLs4CO+XrQbXF57gLotByHfe8eeIiPRLxUma1QCSP7MCt20AHzf5Pgd9Y5V7+1Ui9ZCu3zYm29OS1hSM7rYjWReZL1inbFCohwvckmuuulvzY/ofbnr38QTDplu1eUmvwBUEsBAhQAFAAAAAgA4XsNXVZk1IUaCAAA4xEAAAkAAAAAAAAAAAAAAIABAAAAAFJFQURNRS5tZFBLAQIUABQAAAAIAOF7DV0oi7cjRAAAAEkAAAAIAAAAAAAAAAAAAACAAUEIAAB0cmFpbi5weVBLAQIUABQAAAAIAOF7DV0Pr4zJ/gQAAJcMAAAQAAAAAAAAAAAAAACAAasIAABhc3N1bXB0aW9ucy55YW1sUEsBAhQAFAAAAAgA4XsNXc7EHNSTAwAAEwcAABIAAAAAAAAAAAAAAIAB1w0AAHBhcGVyX2FsaWdubWVudC5tZFBLAQIUABQAAAAIAOF7DV3BZoi3TwAAAFUAAAAQAAAAAAAAAAAAAACAAZoRAAByZXF1aXJlbWVudHMudHh0UEsBAhQAFAAAAAgA4XsNXSZNXZPRAgAA+AcAABcAAAAAAAAAAAAAAIABFxIAAHRyYWNlYWJpbGl0eV9tYXRyaXguY3N2UEsBAhQAFAAAAAgA4XsNXeZDuFGuBAAA4AkAABEAAAAAAAAAAAAAAIABHRUAAGNvbmZpZ3MvYmFzZS55YW1sUEsBAhQAFAAAAAgA4XsNXYgBvYHVAAAAhwEAABsAAAAAAAAAAAAAAIAB+hkAAGNvbmZpZ3MvcGFwZXJfZmFpdGhmdWwueWFtbFBLAQIUABQAAAAIAOF7DV00lQ1OsQAAAEkBAAAfAAAAAAAAAAAAAACAAQgbAABjb25maWdzL3ByYWN0aWNhbF9iYXNlbGluZS55YW1sUEsBAhQAFAAAAAgA4XsNXer/t2JFAAAARQAAAA8AAAAAAAAAAAAAAIAB9hsAAHNyYy9fX2luaXRfXy5weVBLAQIUABQAAAAIAOF7DV2KBfdF4gUAAIwRAAANAAAAAAAAAAAAAACAAWgcAABzcmMvY29uZmlnLnB5UEsBAhQAFAAAAAgA4XsNXbJmmGeyEgAAJEgAAAsAAAAAAAAAAAAAAIABdSIAAHNyYy9kYXRhLnB5UEsBAhQAFAAAAAgA4XsNXWJb6gCIDQAADDIAABYAAAAAAAAAAAAAAIABUDUAAHNyYy9ncmFwaF9zZXF1ZW5jZXMucHlQSwECFAAUAAAACADhew1d1o64OFsHAABjHAAADAAAAAAAAAAAAAAAgAEMQwAAc3JjL21vZGVsLnB5UEsBAhQAFAAAAAgA4XsNXRbBBQLhEQAAKEgAABQAAAAAAAAAAAAAAIABkUoAAHNyYy9wcmVwcm9jZXNzaW5nLnB5UEsBAhQAFAAAAAgA4XsNXbaUD/BYCgAAVCIAAA0AAAAAAAAAAAAAAIABpFwAAHNyYy9zcGxpdHMucHlQSwECFAAUAAAACADhew1dVBAmQDUFAADOEAAAEgAAAAAAAAAAAAAAgAEnZwAAc3JjL3N0ZXAyX3Ntb2tlLnB5UEsBAhQAFAAAAAgA4XsNXRz+juZGBwAAuBcAABIAAAAAAAAAAAAAAIABjGwAAHNyYy9zdGVwM19zbW9rZS5weVBLAQIUABQAAAAIAOF7DV1WauK+agcAADcYAAASAAAAAAAAAAAAAACAAQJ0AABzcmMvc3RlcDRfdHJhaW4ucHlQSwECFAAUAAAACADhew1dkKbO134TAACtRQAADwAAAAAAAAAAAAAAgAGcewAAc3JjL3RyYWluaW5nLnB5UEsBAhQAFAAAAAgA4XsNXVne50KyAwAASQoAABIAAAAAAAAAAAAAAIABR48AAHRlc3RzL3Rlc3RfZGF0YS5weVBLAQIUABQAAAAIAOF7DV2QRW/zLAYAAHsSAAAdAAAAAAAAAAAAAACAASmTAAB0ZXN0cy90ZXN0X2dyYXBoX3NlcXVlbmNlcy5weVBLAQIUABQAAAAIAOF7DV2r+Lex8QEAAMYDAAATAAAAAAAAAAAAAACAAZCZAAB0ZXN0cy90ZXN0X21vZGVsLnB5UEsBAhQAFAAAAAgA4XsNXav6RP/+BAAANQ4AABsAAAAAAAAAAAAAAIABspsAAHRlc3RzL3Rlc3RfcHJlcHJvY2Vzc2luZy5weVBLAQIUABQAAAAIAOF7DV3uv6dVkgIAAJoGAAAUAAAAAAAAAAAAAACAAemgAAB0ZXN0cy90ZXN0X3NwbGl0cy5weVBLBQYAAAAAGQAZAEMGAACtowAAAAA="

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_ARCHIVE_B64))) as project_zip:
    project_zip.extractall(PROJECT_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

if MOUNTED_DATA_DIR.exists():
    DATA_DIR = MOUNTED_DATA_DIR
else:
    DATA_DIR = DOWNLOADED_DATA_DIR
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True)
    download_env = os.environ.copy()
    secret_value = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    try:
        classic = json.loads(secret_value)
    except (TypeError, json.JSONDecodeError):
        download_env["KAGGLE_API_TOKEN"] = secret_value
    else:
        download_env["KAGGLE_USERNAME"] = classic["username"]
        download_env["KAGGLE_KEY"] = classic["key"]
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", "dungnguyen28101991/cicddos2019-parquet",
         "-p", str(DATA_DIR), "--unzip", "--quiet"],
        env=download_env,
        check=True,
    )
    for key in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        download_env.pop(key, None)
    del secret_value
print(f"Step 4 project ready; using dataset at {DATA_DIR}")


In [ ]:
command = [
    sys.executable, "train.py",
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--config", "configs/base.yaml",
    "--mode-config", "configs/practical_baseline.yaml",
    "--samples-per-file", "2048",
    "--sequence-length", "16",
    "--sequence-stride", "8",
    "--epochs", "2",
    "--batch-size", "64",
    "--run-name", "kaggle-step4-smoke",
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)


In [ ]:
summary_path = OUTPUT_DIR / "step4_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert summary["status"] == "passed", summary
assert summary["sequence_leakage_status"] == "passed", summary
assert summary["best_epoch"] in (1, 2), summary
assert (OUTPUT_DIR / "training" / "best_model.pt").exists()
assert (OUTPUT_DIR / "training" / "test_metrics.json").exists()
summary
